# k-means|| vs k-means++ (seriale) vs Random — Confronto (todo #4)

Notebook dedicato al confronto tra k-means|| e i due baseline seriali usati
nel paper di Bahmani et al. (*Scalable K-Means++*, VLDB 2012): k-means++
classico e inizializzazione Random. Stesso 10% di KDDCup99 già usato in
`analysis.ipynb`, per restare direttamente confrontabile con quel notebook.

Moduli usati:
- `kmeans_parallel.py` — algoritmo k-means|| distribuito (invariato)
- `kmeans_serial.py` — k-means++ e Random seriali (nuovo)
- `kmeans_comparison.py` — driver di confronto a tre vie (nuovo)
- `comparison_analysis.py` — tabelle e grafici dedicati al confronto (nuovo)
- `launch_cluster.py`, `data_loader.py`, `benchmark.py` — invariati, riusati così come sono


## 1. Import

In [ ]:
import time
import numpy as np
import pandas as pd

from src.kmeans_parallel import kmeans_parallel
from src.kmeans_serial import kmeans_serial
from src.kmeans_comparison import run_comparison, pilot_timing_check, _materialize_bag
from src.comparison_analysis import summarize_comparison, format_paper_table, plot_cost_by_method, plot_cost_vs_rounds

from src.launch_cluster import launch_cluster, shutdown_cluster
from src.data_loader import load_dataset
from src.benchmark import calculate_inertia


## 2. Parametri configurabili

Cluster e dataset: stessa configurazione di `analysis.ipynb`, per lavorare
sugli stessi dati standardizzati. `MAX_ITER_FIT`/`TOL` sono più generosi
del default usato negli sweep di partizionamento (`max_iter_fit=10`),
perché qui vogliamo il costo *a convergenza* per tutti e tre i metodi, non
solo un budget fisso per confrontare velocità.

In [ ]:
# --- Cluster ---
N_WORKERS = 8      # tra 1 e 8 (nodi disponibili in launch_cluster.py)
NUM_PARTITIONS = 8 * N_WORKERS   # regola empirica: >= n_threads_per_worker * n_workers

# --- Confronto ---
MAX_ITER_FIT = 100          # budget generoso: vogliamo il costo a convergenza, non solo veloce
TOL = 1e-4
SEED = 42
AVERAGING_ITERATIONS = 11   # convenzione del paper: mediana su 11 run per le tabelle di costo
N_LOCAL_TRIALS = None       # None = k-means++ greedy (default sklearn); 1 = Algoritmo 1 "vanilla" del paper


In [ ]:
# --- Dataset (identico a analysis.ipynb, sezione 2) ---
DATASET_URL_10PC = "https://ndownloader.figshare.com/files/5976042"
# link used by sklearn function fetch_kddcup99 (original link gives 403 error)
#***for 10% dataset***

DATASET_URL_FULL = "https://ndownloader.figshare.com/files/5976045"
#***FULL DATASET***

RAW_GZ_PATH  = "/home/ubuntu/Project/libero_development/data/kddcup_data.gz"
PARQUET_PATH = '/tmp/kddcup_data.parquet'

COL_NAMES = [
    "duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot",
    "num_failed_logins","logged_in","num_compromised","root_shell",
    "su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate",
    "dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate",
    "dst_host_srv_rerror_rate","label"
]


## 3. Avvio/connessione al cluster

In [ ]:
# DO NOT RUN if already existing!
cluster, client = launch_cluster(N_WORKERS)


#### IF INSTEAD ALREADY EXISTING CLUSTER:

In [ ]:
from dask.distributed import Client

SCHEDULER_ADDRESS = "tcp://10.67.22.194:8786"

try:
    client = Client(SCHEDULER_ADDRESS, timeout="10s")
    print("Connected to cluster successfully!")
    print(f"Dask Dashboard link: {client.dashboard_link}")

except Exception as e:
    print(f"Connection error: {e}")


In [ ]:
client.scheduler_info()


## 4. Caricamento dataset (stesso 10% KDDCup99 di `analysis.ipynb`)

In [ ]:
start = time.time()
X_bag, (mean_ar, std_ar) = load_dataset(
    n_partitions=NUM_PARTITIONS,
    client=client,
    dataset_url=DATASET_URL_10PC,
    raw_gz_path=RAW_GZ_PATH,
    parquet_path=PARQUET_PATH,
    parquet_path_workers=PARQUET_PATH,
    col_names=COL_NAMES,
)
elapsed = time.time() - start
print(f"Time elapsed: {elapsed:.2f} s")


## 5. Verifica di fattibilità

Il seeding *greedy* di k-means++ (default di sklearn) su ~494k righe può
essere lento. Prima di lanciare il loop completo del confronto, cronometra
una singola run (seed + fit) al `k` più piccolo che hai intenzione di
testare qui sotto.

In [ ]:
K_SMALLEST = 500  # deve coincidere con il valore più piccolo in K_VALUES (sezione 6)

X_arr = _materialize_bag(client, X_bag)

for init in ("k-means++", "random"):
    pilot_timing_check(
        X_arr, k=K_SMALLEST, init=init, seed=SEED,
        n_local_trials=N_LOCAL_TRIALS, max_iter=MAX_ITER_FIT, tol=TOL,
    )


**Punto di decisione**: moltiplica il tempo totale per
`AVERAGING_ITERATIONS` e per il numero di valori in `K_VALUES`. Se non è
accettabile, in ordine di preferenza:

1. riduci `AVERAGING_ITERATIONS` solo per il lato seriale (nel codice di
   `kmeans_comparison.run_comparison` — richiede una piccola modifica se
   vuoi ripetizioni diverse tra k-means|| e i baseline seriali);
2. come ultima risorsa, sottocampiona ulteriormente **sia** `X_arr` sia
   `X_bag` usati nel confronto — altrimenti i costi non sono più
   confrontabili in valore assoluto tra i tre metodi.

## 6. Configurazione del confronto

`K_VALUES` e `parallel_combinations` sono da compilare con le tue scelte.
Se in `parallel_combinations` includi più valori di `r` alla stessa
`l_over_k`, la Sezione 8 può anche riprodurre il grafico costo-vs-round in
stile Figura 5.2/5.3 del paper.

In [ ]:
K_VALUES = [500, 1000]  # <- scegli qui i valori di k da testare

parallel_combinations = [
    # (num_partitions, l_over_k, r) — scegli qui le combinazioni k-means|| da confrontare
    (NUM_PARTITIONS, 1.0, 5),
    (NUM_PARTITIONS, 2.0, 5),
]


## 7. Esecuzione del confronto

In [ ]:
df_comparison = run_comparison(
    client, X_bag,
    k_values=K_VALUES,
    parallel_combinations=parallel_combinations,
    seed=SEED,
    averaging_iterations=AVERAGING_ITERATIONS,
    max_iter_fit=MAX_ITER_FIT,
    tol=TOL,
    n_local_trials=N_LOCAL_TRIALS,
    label="kmeans_comparison",
)
df_comparison


## 8. Analisi

Tabella in stile Tabelle 1/2 del paper (mediana su `AVERAGING_ITERATIONS`
run, seed/final affiancati), dettagli mean±std, e bar chart per confrontare
le categorie (i tre metodi) per `k`.

In [ ]:
print(format_paper_table(df_comparison, stat="median"))


In [ ]:
summarize_comparison(df_comparison)


In [ ]:
plot_cost_by_method(df_comparison, metric="cost_seed")
plot_cost_by_method(df_comparison, metric="cost_final")


Grafico costo-vs-round (Figura 5.2/5.3-style) — solo se in
`parallel_combinations` hai incluso più valori di `r` alla stessa `l_over_k`
per uno dei `K_VALUES`:

In [ ]:
# plot_cost_vs_rounds(df_comparison, k=500, metric="cost_final")


**Nota sui tempi**: `time_seed`/`time_fit` in `df_comparison` sono
riportati solo per riferimento, non come confronto controllato —
`kmeans_serial` gira single-thread sul processo client, `kmeans_parallel`
gira distribuito sul cluster Dask. Per un confronto di wall-clock corretto
servirebbe hardware comparabile (es. un solo nodo del cluster), fuori
dallo scope di questo notebook. L'asse primario di confronto qui è la
qualità (`cost_seed`/`cost_final`), come nel paper.

## 9. Spegnimento del cluster

Da eseguire a fine lavoro, o prima di rilanciare `launch_cluster` con un
`N_WORKERS` diverso.

In [ ]:
shutdown_cluster(cluster, client)
